# Basic agent

In [ ]:
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
print(load_dotenv())

C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


In the following example you will build a research agent that can answer questions about text files. Along the way you will explore the following concepts:
1. Detailed system prompts for better agent behavior
2. Create tools that integrate with external data
3. Model configuration for consistent responses
4. onversational memory for chat-like interactions
5. Deep Agents for built-in features
6. Testing your agent

In [ ]:
%%time
# Tools
def get_weather(city: str) -> str:
    '''
    
    '''
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openrouter:openai/gpt-oss-120b:free",
    tools=[get_weather],
    system_prompt="You are a AI assistant",
)

response = agent.invoke({
    'message': [{
        'role': 'user',
        'content': "What's the weather in San Francisco?",
    }]
})
# print(response["messages"])

CPU times: total: 375 ms
Wall time: 3.51 s


In [52]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openrouter:openai/gpt-oss-120b:free",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content)

San Francisco is currently sunny. Enjoy the clear skies!


## 1. Define a system prompt

In [53]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

## 2. Create tools

Runtime context

Agent memory

In [54]:
import urllib.error
import urllib.request

from langchain.tools import tool

@tool # Nhớ phải có thằng này
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

## 3. Configure your model

In [58]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openrouter:openai/gpt-oss-120b:free",
    temperature=0.5,
    timeout=300,
    max_tokens=25000,
)

## 4. Add memory 

In [57]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 5. Create and run the agent

There are two different frameworks for creating agents: **LangChain agents** and **deep agents**

In [60]:
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print(agent_result["messages"][-1].content_blocks)
print("\n")
print(deep_agent_result["messages"][-1].content_blocks)

ReadTimeout: The read operation timed out

The deep agent, on the other hand can:
1. Plans its approach using the built-in write_todos tool to break down the research task -> like task planning
2. Loads the file by calling the fetch_text_from_url tool to gather information.
3. Manages context by using file system tools (grep and read_file).
4. Spawns subagents as needed to delegate complex subtasks to specialized subagents.

# Trace agent calls

In [ ]:
# export LANGSMITH_TRACING="true"
# export LANGSMITH_API_KEY="..."